In [21]:
# ====================
# Imports and Setup
# ====================
# No installs needed if previous notebooks ran

import random
import pandas as pd
import numpy as np

from IPython.display import display, HTML
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [22]:
# -----------------------------
# Load Notebook One Artifacts
# -----------------------------

df = pd.read_csv("rag_base_dataset.csv")
embeddings = np.load("essay_embeddings.npy")

In [23]:
# ====================
# Styling Layer
# ====================
def styled_container(content):
    return f"""
    <div style="
        background-color: #000000;
        color: #FFFFFF;
        padding: 25px;
        border-radius: 10px;
        font-family: 'Times New Roman', serif;
        line-height: 1.6;
    ">
        {content}
    </div>
    """

In [24]:
# ==============================
# Section Builder
# ==============================
# Build the display Elements
# Calls Styling Elements
# Creates clean display for RAG
# ==============================

# Title Block
def render_title():
    return """
    <h1 style="color:#8B0000; text-align:center;">
        Essay Evaluation Report
    </h1>
    <hr style="border: 1px solid #5C4033;">
    """

# Score Display
def render_score(score):
    return f"""
    <h2 style="color:#D4AF37;">
        Final Score: {score}
    </h2>
    """

# Feedback Section
def render_feedback(feedback_list):

    items = "".join([f"<li>{f}</li>" for f in feedback_list])

    return f"""
    <h3 style="color:#8B0000;">📝 Feedback</h3>
    <ul style="color:#F5F5DC;">
        {items}
    </ul>
    """

# Suggestions Section
def render_suggestions(suggestions):

    items = "".join([f"<li>{s}</li>" for s in suggestions])

    return f"""
    <h3 style="color:#8B0000;">📚 Improvement Suggestions</h3>
    <ul style="color:#F5F5DC;">
        {items}
    </ul>
    """

# Explanation Section
def render_explanation(explanation):

    items = "".join([f"<li>{e}</li>" for e in explanation])

    return f"""
    <h3 style="color:#8B0000;">🔍 Grade Explanation</h3>
    <ul style="color:#F5F5DC;">
        {items}
    </ul>
    """

# Teacher Commnet Banner
def teacher_comment():
    comments = [
        "Keep pushing—you're close to a strong essay.",
        "Good effort, but focus on deeper analysis.",
        "Nice structure—refine your argument further."
    ]

    return f"""
    <p style="color:#D4AF37; font-style:italic;">
        Teacher Comment: {random.choice(comments)}
    </p>
    """

In [25]:
# ==========================
# Recreate Feature Pipeline
# ==========================

def vocab_richness(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

df["word_count"] = df["full_text"].astype(str).apply(lambda x: len(x.split()))
df["vocab_richness"] = df["full_text"].astype(str).apply(vocab_richness)

feature_cols = ["word_count", "vocab_richness"]
X_features = df[feature_cols].fillna(0)

scaler = MinMaxScaler()
X_features_scaled = scaler.fit_transform(X_features)

# -------------------
# Retrieval Model
# -------------------

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
# ========================
# Retrieval Function
# ========================

def retrieve_similar_essays(new_text, top_k=5):

    new_embedding = model.encode([new_text])
    semantic_sim = cosine_similarity(new_embedding, embeddings)[0]

    new_word_count = len(new_text.split())
    new_vocab = vocab_richness(new_text)

    new_features_df = pd.DataFrame(
      [[new_word_count, new_vocab]],
      columns=feature_cols
    )

    new_features = scaler.transform(new_features_df)
    feature_sim = cosine_similarity(new_features, X_features_scaled)[0]

    # UPDATED WEIGHTS
    final_score = 0.6 * semantic_sim + 0.4 * feature_sim

    top_indices = np.argsort(final_score)[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results["semantic_similarity"] = semantic_sim[top_indices]
    results["feature_similarity"] = feature_sim[top_indices]
    results["final_score"] = final_score[top_indices]

    return results

In [27]:
# ======================
# Core RAG Engine
# ======================

# This is where the fun begins!

# ------------------
# Helper Functions
# ------------------

# Extract Writing Signals
def analyze_essay(text):
    sentences = text.split(".")

    return {
        "length": len(text.split()),
        "sentences": len(sentences),
        "avg_sentence_length": len(text.split()) / max(len(sentences), 1),
        "vocab_richness": vocab_richness(text)
    }


# Compare to high-scoring Essays
def compare_to_top_essays(query_text, retrieved_df):

    query_stats = analyze_essay(query_text)

    high_score_df = retrieved_df[retrieved_df["score"] >= 4]

    if len(high_score_df) == 0:
        return "Not enough high-scoring essays to compare."

    avg_word_count = high_score_df["word_count"].mean()
    avg_vocab = high_score_df["vocab_richness"].mean()

    feedback = []

    if query_stats["length"] < avg_word_count:
        feedback.append("Your essay is shorter than higher-scoring essays. Consider expanding your ideas.")

    if query_stats["vocab_richness"] < avg_vocab:
        feedback.append("Try using more varied vocabulary to strengthen your writing.")

    if query_stats["avg_sentence_length"] < 10:
        feedback.append("Your sentences are quite short. Try combining ideas for more complex structure.")

    return feedback


In [28]:
# ==========================================
# RAG Interpretation Modalities
# ==========================================
# Responses are pulled from library
# Stochastic phrase selection
# Simulates LLM response from context
# There are four modes to this RAG model
# Mode One: Feedback Generator
# Mode Two: Grade Explanation
# Mode Three: Improvement Suggestions
# Mode Four: Simple Auto Scoring
# ===========================================

# ---------------------------
# Phrase Pool Implementation
# ---------------------------

# Create Phrase Pool
import random

FEEDBACK_LIBRARY = {
    "length_short": [
        "Your essay is shorter than stronger examples—try expanding your ideas.",
        "Consider adding more detail to match higher-scoring essays.",
        "Your response could benefit from more development and depth."
    ],

    "low_vocab": [
        "Try using more varied vocabulary to strengthen your writing.",
        "Your word choice is somewhat repetitive—consider diversifying it.",
        "Using a wider range of vocabulary could improve your essay."
    ],

    "good": [
        "You're on the right track and close to stronger responses.",
        "This essay shows solid understanding but could be refined further.",
        "You have a good foundation—just a few improvements needed."
    ],

    "weak": [
        "This response aligns more with lower-scoring essays.",
        "Your essay may need significant improvement to reach higher levels.",
        "There are several areas that need development."
    ]
}


# Mode One
def generate_feedback(query_text):

    retrieved = retrieve_similar_essays(query_text)
    query_stats = analyze_essay(query_text)

    feedback = []

    avg_score = retrieved["score"].mean()

    # Score-based feedback
    if avg_score < 3:
        feedback.append(random.choice(FEEDBACK_LIBRARY["weak"]))
    elif avg_score < 4:
        feedback.append(random.choice(FEEDBACK_LIBRARY["good"]))
    else:
        feedback.append("This essay is similar to high-scoring responses.")

    # Compare to high-scoring essays
    high_score_df = retrieved[retrieved["score"] >= 4]

    if len(high_score_df) > 0:
        avg_word_count = high_score_df["word_count"].mean()
        avg_vocab = high_score_df["vocab_richness"].mean()

        diff = avg_word_count - query_stats["length"]

        if diff > 50:
            feedback.append(f"Your essay is about {int(diff)} words shorter than strong examples.")
            feedback.append(random.choice(FEEDBACK_LIBRARY["length_short"]))
        elif diff > 20:
            feedback.append("Your essay is slightly shorter than higher-scoring responses.")

        if query_stats["vocab_richness"] < avg_vocab:
            feedback.append(
                f"Your vocabulary richness ({query_stats['vocab_richness']:.2f}) "
                f"is below stronger essays (~{avg_vocab:.2f})."
            )
            feedback.append(random.choice(FEEDBACK_LIBRARY["low_vocab"]))

    return {
        "retrieved": retrieved,
        "feedback": feedback
    }

# Mode Two
def explain_grade(query_text):

    retrieved = retrieve_similar_essays(query_text)

    explanation = []

    explanation.append(
        f"Your essay is most similar to essays scoring between "
        f"{retrieved['score'].min()} and {retrieved['score'].max()}."
    )

    explanation.append(
        f"The average similarity-based score is approximately "
        f"{retrieved['score'].mean():.2f}."
    )

    explanation.append(
        "This suggests your writing level aligns with these examples "
        "in both content and structure."
    )

    return explanation


# Mode Three
def suggest_improvements(query_text):

    retrieved = retrieve_similar_essays(query_text)
    query_stats = analyze_essay(query_text)

    suggestions = []

    high_score_df = retrieved[retrieved["score"] >= 5]

    if len(high_score_df) == 0:
        return ["No top-tier examples found. Try expanding your essay."]

    avg_word_count = high_score_df["word_count"].mean()
    avg_vocab = high_score_df["vocab_richness"].mean()

    # Targeted improvements
    if query_stats["length"] < avg_word_count:
        suggestions.append("Expand your essay to include more detailed arguments and examples.")

    if query_stats["vocab_richness"] < avg_vocab:
        suggestions.append("Incorporate more varied and precise vocabulary.")

    if query_stats["avg_sentence_length"] < 10:
        suggestions.append("Try combining sentences to create more complex and fluid writing.")

    # General high-score traits
    suggestions.append("Top-scoring essays tend to:")
    suggestions.append("- Present clear and well-supported arguments")
    suggestions.append("- Use specific and relevant examples")
    suggestions.append("- Maintain strong organization across paragraphs")

    return suggestions


# Mode Four
def predict_score(query_text):

    retrieved = retrieve_similar_essays(query_text)

    return float(round(retrieved["score"].mean(), 2))

# ----------------------------------------
# Paragraph Builder (Final Output Layer)
# ----------------------------------------

def build_paragraph(feedback_list):

    intro_options = [
        "Here's some feedback on your essay:",
        "After reviewing similar essays, here's what stands out:",
        "Based on comparable responses, here are some suggestions:"
    ]

    intro = random.choice(intro_options)

    paragraph = intro + "\n\n"

    for f in feedback_list:
        paragraph += f"- {f}\n"

    return paragraph

In [29]:
# ==================
# Unified Interface
# ==================

def rag_system(query_text, mode="feedback"):

    if mode == "feedback":
        result = generate_feedback(query_text)
        return result

    elif mode == "explain":
        return explain_grade(query_text)

    elif mode == "improve":
        return suggest_improvements(query_text)

    elif mode == "score":
        return predict_score(query_text)

    else:
        return "Invalid mode."

In [30]:
# =======================
# Final Displat Function
# =======================

def display_full_report(query_text):

    feedback_result = rag_system(query_text, mode="feedback")
    explanation = rag_system(query_text, mode="explain")
    suggestions = rag_system(query_text, mode="improve")
    score = rag_system(query_text, mode="score")

    content = ""

    content += render_title()
    content += render_score(score)
    content += render_feedback(feedback_result["feedback"])
    content += render_suggestions(suggestions)
    content += render_explanation(explanation)
    content += teacher_comment()

    display(HTML(styled_container(content)))

In [31]:
# ----------
# Demo Run
# ---------

test_essay = df["full_text"].iloc[10]

display_full_report(test_essay)